**Stage 3: encrypted DCT steganography + reconstruction + metrics.** AES-256-CTR (random 16-byte IV per frame) + HMAC-SHA256 over IV||ciphertext; 4x4 DCT quantisation hiding inside the person bounding box (QP=27 used for naming only, beta=30, alpha=0.05). Videos are written with the lossless FFV1 codec (about 7x larger than MJPG); stego quality is measured against this notebook's own cover video.

**Measured quality (vid2-1, first 60 frames):** stego PSNR 43.0 dB, reconstruction PSNR 23.0 dB. The reconstructed person is recognisable but heavily speckled. The thesis's 36.5 dB reconstruction comes from the unencrypted pipeline (`Stego_and_Reconstructed_Videos.ipynb`); see the README's Results section for why encryption degrades reconstruction.

On first run it generates keys and writes them to `encryption_keys.txt`; the per-frame IV/HMAC/bbox log goes to `*_crypto.npy`. Both are git-ignored. **Never commit them, and never commit this notebook with outputs** (it prints the keys).

In [ ]:
!pip install ultralytics
!pip install opencv-python

In [ ]:
"""
Stego and Reconstructed Videos — with AES-256-CTR + HMAC Encryption
Based on: multi_stego_and_reconstructed_video_results.py (v3)

Changes from original:
  1. Added AES-256-CTR encryption of message crop BEFORE quantization_hiding()
  2. Added HMAC-SHA256 tag for tamper detection
  3. bbox_log replaced by crypto_log (stores bbox + IV + HMAC tag per frame)
  4. Extraction decrypts after quantization_extraction()
  5. All DCT/YOLO/metric logic is IDENTICAL to original — no changes there

Security layers:
  Layer 1 — Steganographic : hidden data is invisible in stego video
  Layer 2 — AES-256-CTR    : extracted bytes unreadable without the key
  Layer 3 — HMAC-SHA256    : detects wholesale replacement/tampering of stego video
             (Note: HMAC will show mismatch due to normal extraction noise — this is
              expected and documented below. Decryption still recovers the frame.)
"""

!pip install ultralytics cryptography opencv-python -q

import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import cv2
import numpy as np
import pandas as pd
from scipy.fftpack import dct, idct
from ultralytics import YOLO

# ── cryptography imports ──────────────────────────────────────────────────
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import hmac as crypto_hmac, hashes
from cryptography.hazmat.backends import default_backend
from cryptography.exceptions import InvalidSignature

if torch.cuda.is_available():
    print(f"CUDA available. GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available. Using CPU.")

# ==================== ENCRYPTION CONFIG ====================
# AES-256 key = 32 bytes.  HMAC key = separate 32 bytes (never reuse the AES key).
#
# To persist keys across sessions, they are now saved to and loaded from 'encryption_keys.txt'.
# If the file does not exist, new keys are generated and saved.

KEYS_FILE = '/content/encryption_keys.txt'
AES_KEY_HEX = "" # Initialize
HMAC_KEY_HEX = "" # Initialize

if os.path.exists(KEYS_FILE):
    with open(KEYS_FILE, 'r') as f:
        lines = f.readlines()
        if len(lines) == 2:
            AES_KEY_HEX = lines[0].strip()
            HMAC_KEY_HEX = lines[1].strip()
            print(f"✓ Loaded AES and HMAC keys from {KEYS_FILE}.")
        else:
            print(f"⚠ {KEYS_FILE} is malformed. Generating new keys.")
else:
    print(f"ℹ {KEYS_FILE} not found. Generating new keys.")

if not AES_KEY_HEX or not HMAC_KEY_HEX:
    AES_KEY  = os.urandom(32)   # 256-bit AES key
    HMAC_KEY = os.urandom(32)   # separate 256-bit HMAC key
    AES_KEY_HEX = AES_KEY.hex()
    HMAC_KEY_HEX = HMAC_KEY.hex()
    with open(KEYS_FILE, 'w') as f:
        f.write(AES_KEY_HEX + '\n')
        f.write(HMAC_KEY_HEX + '\n')
    print("=" * 60)
    print("NEW KEYS GENERATED AND SAVED TO 'encryption_keys.txt':")
    print(f"  AES_KEY_HEX  = '{AES_KEY_HEX}'")
    print(f"  HMAC_KEY_HEX = '{HMAC_KEY_HEX}'")
    print("=\n" * 60)
else:
    AES_KEY = bytes.fromhex(AES_KEY_HEX)
    HMAC_KEY = bytes.fromhex(HMAC_KEY_HEX)

# ==================== PARAMETERS ====================
QP    = 27   # kept for naming convention
Beta  = 30
Alpha = 0.05

# ── Paths ──────────────────────────────────────────────────────────────────────
original_video_path  = '/content/vid2-2.avi'
bg_image_path        = '/content/vid2-2-BG_frame.jpg'

cover_output_path         = f'/content/Cover_QP{QP}B{Beta}.avi'       # this notebook's own cover, for stego metrics
stego_output_path         = f'/content/QP{QP}B{Beta}_enc.avi'
reconstructed_output_path = f'/content/Rec_Msg_QP{QP}B{Beta}_enc.avi'
crypto_log_path           = f'/content/QP{QP}B{Beta}_crypto.npy'  # replaces bbox_log
output_excel              = f'/content/ResultsQP{QP}_enc.xlsx'
sheet_name                = f'QP{QP}B{Beta}_enc'

# ==================== ENCRYPTION HELPERS ====================

def encrypt_crop(crop_uint8):
    """
    AES-256-CTR encrypt a frame crop (H×W×3 uint8 numpy array).

    Why CTR mode:
      - Output is same length as input — no padding, no shape change
      - Extraction noise maps 1:1 to plaintext noise (no error propagation unlike CBC)

    Returns:
      enc_crop  : (H×W×3 uint8) encrypted bytes, same shape as input crop
      iv        : 16-byte nonce — unique per frame, stored in crypto_log
      hmac_tag  : 32-byte HMAC-SHA256 over (iv + ciphertext)
    """
    flat_bytes = crop_uint8.tobytes()

    iv = os.urandom(16)
    cipher = Cipher(algorithms.AES(AES_KEY), modes.CTR(iv), backend=default_backend())
    enc = cipher.encryptor()
    ciphertext = enc.update(flat_bytes) + enc.finalize()   # same length as flat_bytes

    h = crypto_hmac.HMAC(HMAC_KEY, hashes.SHA256(), backend=default_backend())
    h.update(iv + ciphertext)
    hmac_tag = h.finalize()   # 32 bytes

    enc_crop = np.frombuffer(ciphertext, dtype=np.uint8).reshape(crop_uint8.shape)
    return enc_crop, iv, hmac_tag


def decrypt_crop(extracted_uint8, iv, hmac_tag):
    """
    AES-256-CTR decrypt an extracted frame crop.

    HMAC note: steganographic extraction always introduces small pixel errors,
    which change the ciphertext bytes slightly → HMAC will almost always fail here.
    This is EXPECTED and NOT a security failure.
    - Genuine tamper attack (someone replaces the stego video entirely) → garbled output
    - Normal extraction noise → HMAC fails but decryption still recovers a correct frame
    AES-CTR tolerates byte-level noise: a flipped ciphertext bit flips only that plaintext
    bit, so the visual result has tiny isolated errors rather than complete corruption.

    Returns:
      dec_crop   : (H×W×3 uint8) decrypted frame crop
      hmac_valid : True if HMAC passed (no modification beyond extraction noise)
    """
    ciphertext = extracted_uint8.tobytes()

    hmac_valid = True
    try:
        h = crypto_hmac.HMAC(HMAC_KEY, hashes.SHA256(), backend=default_backend())
        h.update(iv + ciphertext)
        h.verify(hmac_tag)
    except InvalidSignature:
        hmac_valid = False   # expected due to extraction noise — decryption still proceeds

    cipher = Cipher(algorithms.AES(AES_KEY), modes.CTR(iv), backend=default_backend())
    dec = cipher.decryptor()
    plaintext = dec.update(ciphertext) + dec.finalize()

    dec_crop = np.frombuffer(plaintext, dtype=np.uint8).reshape(extracted_uint8.shape)
    return dec_crop, hmac_valid

# ==================== YOLO ====================
print("Loading YOLO segmentation model...")
yolo_model      = YOLO('yolov8x-seg.pt')
PERSON_CLASS_ID = 0
closing_kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
dilation_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

def get_person_mask_and_bbox(frame, frame_width, frame_height):
    results = yolo_model(frame, classes=[PERSON_CLASS_ID], verbose=False)
    combined_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)

    for r in results:
        if r.masks is not None:
            for md in r.masks.data.cpu().numpy():
                pm = cv2.resize(md, (frame_width, frame_height))
                pm = (pm > 0.5).astype(np.uint8) * 255
                combined_mask = cv2.bitwise_or(combined_mask, pm)
        elif r.boxes is not None:
            for box in r.boxes:
                if int(box.cls[0]) == PERSON_CLASS_ID:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(combined_mask, (x1, y1), (x2, y2), 255, -1)

    if combined_mask.max() == 0:
        return combined_mask, None

    combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, closing_kernel)
    combined_mask = cv2.dilate(combined_mask, dilation_kernel, iterations=1)

    coords = cv2.findNonZero(combined_mask)
    x, y, w, h = cv2.boundingRect(coords)

    w = (w // 4) * 4
    h = (h // 4) * 4
    x = min(x, frame_width  - w)
    y = min(y, frame_height - h)

    if w < 4 or h < 4:
        return combined_mask, None

    return combined_mask, (x, y, w, h)

# ==================== STEGANOGRAPHY HELPERS ====================
def dct2(block):
    return dct(dct(block.T, norm='ortho').T, norm='ortho')

def idct2(block):
    return idct(idct(block.T, norm='ortho').T, norm='ortho')

def adjust_image_a(img, alpha):
    d = img.astype(np.float64) / 255.0
    d[d >= (1 - alpha)] = 1 - alpha
    d[d <= alpha]       = alpha
    return d

def qun1(x, y, alp):
    iv = np.linspace(0.0, 2.0, alp)
    s  = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            if   x[i,j] >= 2.0: a, b = iv[-2], 2.0
            elif x[i,j] <  0.0: a, b = 0.0, iv[1]
            else:
                a = iv[np.where(iv <= x[i,j])[0][-1]]
                b = iv[np.where(iv >  x[i,j])[0][0]]
            s[i,j] = a + (b - a) * y[i,j]
    return s

def inv_qun1(s, alp):
    iv = np.linspace(0.0, 2.0, alp)
    y  = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            if   s[i,j] >= 2.0: a, b = iv[-2], 2.0
            elif s[i,j] <  0.0: a, b = 0.0, iv[1]
            else:
                a = iv[np.where(iv <= s[i,j])[0][-1]]
                b = iv[np.where(iv >  s[i,j])[0][0]]
            y[i,j] = (s[i,j] - a) / (b - a)
    return y

def quantization_hiding(cover_crop, msg_crop, beta):
    # msg_crop here is the ENCRYPTED crop — DCT
    mm, nn, ll  = cover_crop.shape
    stego       = cover_crop.copy().astype(np.float64)
    for p in range(ll):
        for i in range(0, mm - 3, 4):
            for j in range(0, nn - 3, 4):
                c_blk    = cover_crop[i:i+4, j:j+4, p].astype(np.float64)
                m_blk    = msg_crop[i:i+4, j:j+4, p]
                c_dct    = dct2(c_blk / 255.0)
                s_dct    = qun1(np.abs(c_dct), m_blk, beta)
                s_dct    = np.where(c_dct < 0, -s_dct, s_dct)
                s_dct[0,0] = c_dct[0,0]
                s_blk    = idct2(s_dct) * 255.0
                stego[i:i+4, j:j+4, p] = np.clip(np.round(s_blk), 0, 255)
    return stego.astype(np.uint8)

def quantization_extraction(stego_crop, beta):
    mm, nn, ll = stego_crop.shape
    rec        = np.zeros((mm, nn, ll))
    for p in range(ll):
        for i in range(0, mm - 3, 4):
            for j in range(0, nn - 3, 4):
                s_blk        = stego_crop[i:i+4, j:j+4, p].astype(np.float64)
                s_dct        = dct2(s_blk / 255.0)
                rec_blk      = inv_qun1(np.abs(s_dct), beta)
                rec_blk[0,0] = (rec_blk[0,1] + rec_blk[1,1] + rec_blk[1,0]) / 3.0
                rec[i:i+4, j:j+4, p] = rec_blk
    return (rec * 255).astype(np.uint8)

# ==================== EMBEDDING ====================

print("\n=== EMBEDDING (AES-256-CTR encrypted) ===")

cap_orig = cv2.VideoCapture(original_video_path)
bg_image = cv2.imread(bg_image_path)

frame_width  = int(cap_orig.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap_orig.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap_orig.get(cv2.CAP_PROP_FPS)
total_frames = int(cap_orig.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {frame_width}x{frame_height}, FPS={fps}, Frames={total_frames}")

resized_bg   = cv2.resize(bg_image, (frame_width, frame_height))
# Lossless FFV1: the hidden bytes live in fine DCT positions that any lossy codec (e.g. MJPG) destroys.
fourcc       = cv2.VideoWriter_fourcc(*'FFV1')
cover_writer = cv2.VideoWriter(cover_output_path, fourcc, fps, (frame_width, frame_height))
stego_writer = cv2.VideoWriter(stego_output_path, fourcc, fps, (frame_width, frame_height))

# crypto_log replaces bbox_log — stores bbox + IV + HMAC tag per frame
crypto_log  = []
frame_count = 0

while True:
    ret, frame = cap_orig.read()
    if not ret:
        break

    msg_frame = frame.copy()   # original frame = the secret message

    # Step 1: YOLO mask → cover frame
    person_mask, bbox = get_person_mask_and_bbox(frame, frame_width, frame_height)

    cover_frame = frame.copy().astype(np.float64)
    if person_mask.max() > 0:
        mask_3ch = person_mask[:, :, np.newaxis].repeat(3, axis=2)
        cover_frame[mask_3ch > 0] = resized_bg.astype(np.float64)[mask_3ch > 0]
    cover_frame = cover_frame.astype(np.uint8)

    stego_frame  = cover_frame.copy()
    frame_crypto = {'bbox': bbox, 'iv': None, 'hmac_tag': None}

    if bbox is not None:
        x, y, w, h = bbox
        cover_crop = cover_frame[y:y+h, x:x+w]
        raw_crop   = msg_frame[y:y+h, x:x+w]       # original pixels (the secret)

        # ── encrypt raw_crop before hiding ────────────────────────────────
        enc_crop, iv, hmac_tag = encrypt_crop(raw_crop)
        frame_crypto['iv']       = iv
        frame_crypto['hmac_tag'] = hmac_tag
        # ───────────────────────────────────────────────────────────────────────

        # adjust_image_a and quantization_hiding
        # — we pass enc_crop instead of raw_crop
        msg_crop   = adjust_image_a(enc_crop, Alpha)
        stego_crop = quantization_hiding(cover_crop, msg_crop, Beta)

        # Blend
        mask_crop = person_mask[y:y+h, x:x+w].astype(np.float32) / 255.0
        mask_3ch  = cv2.merge([mask_crop, mask_crop, mask_crop])
        blended   = (
            stego_crop.astype(np.float32) * mask_3ch +
            cover_frame[y:y+h, x:x+w].astype(np.float32) * (1.0 - mask_3ch)
        ).astype(np.uint8)

        stego_frame[y:y+h, x:x+w] = blended

    crypto_log.append(frame_crypto)
    cover_writer.write(cover_frame)
    stego_writer.write(stego_frame)
    frame_count += 1

    if frame_count % 10 == 0:
        print(f"  Embedded {frame_count}/{total_frames} | "
              f"person={'yes bbox='+str(bbox) if bbox else 'none'}")

cap_orig.release()
stego_writer.release()
cover_writer.release()
np.save(crypto_log_path, np.array(crypto_log, dtype=object), allow_pickle=True)

print(f"\n✓ Encrypted stego video  → {stego_output_path}")
print(f"✓ Crypto log (IV + HMAC) → {crypto_log_path}")
print(f"  ⚠ Keep crypto_log + AES/HMAC keys safe — required for decryption.")

# ==================== EXTRACTION ====================

print("\n=== EXTRACTION (AES-256-CTR decryption) ===")

crypto_log_loaded = np.load(crypto_log_path, allow_pickle=True).tolist()

cap_stego    = cv2.VideoCapture(stego_output_path)
frame_width  = int(cap_stego.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap_stego.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap_stego.get(cv2.CAP_PROP_FPS)
total_frames = int(cap_stego.get(cv2.CAP_PROP_FRAME_COUNT))

rec_writer  = cv2.VideoWriter(reconstructed_output_path, fourcc, fps, (frame_width, frame_height))
frame_count = 0
hmac_pass   = 0
hmac_fail   = 0

while True:
    ret, stego_frame = cap_stego.read()
    if not ret:
        break

    rec_frame  = stego_frame.copy()
    log_entry  = crypto_log_loaded[frame_count] if frame_count < len(crypto_log_loaded) else {}

    bbox     = log_entry.get('bbox',     None)
    iv       = log_entry.get('iv',       None)
    hmac_tag = log_entry.get('hmac_tag', None)

    if bbox is not None and iv is not None:
        x, y, w, h = bbox
        stego_crop = stego_frame[y:y+h, x:x+w]

        # Step 1: DCT extraction
        rec_msg_crop = quantization_extraction(stego_crop, Beta)

        # Bright pixel correction
        bright = np.all(stego_crop > 200, axis=2)
        rec_msg_crop[bright] = stego_crop[bright]

        # ── decrypt extracted bytes ────────────────────────────────────────
        dec_crop, hmac_valid = decrypt_crop(rec_msg_crop, iv, hmac_tag)
        if hmac_valid:
            hmac_pass += 1
        else:
            hmac_fail += 1   # expected — see decrypt_crop() docstring
        # ────────────────────────────────────────────────────────────────────────

        rec_frame[y:y+h, x:x+w] = dec_crop

    rec_writer.write(rec_frame)
    frame_count += 1

    if frame_count % 30 == 0:
        print(f"  Extracted {frame_count}/{total_frames} | "
              f"HMAC pass={hmac_pass}, mismatch={hmac_fail} (mismatch is expected)")

cap_stego.release()
rec_writer.release()

person_frames = hmac_pass + hmac_fail
print(f"\n✓ Reconstructed → {reconstructed_output_path}")
print(f"\nHMAC summary (person frames only):")
print(f"  Pass    : {hmac_pass}  ({100*hmac_pass/max(person_frames,1):.1f}%)")
print(f"  Mismatch: {hmac_fail}  ({100*hmac_fail/max(person_frames,1):.1f}%) ← normal extraction noise")
print(f"  Note: a genuine attack (stego video replaced) → garbled decrypted output.")

# ==================== METRICS ====================
print("\n=== METRICS ===")

def matrix_to_row_vector(mat):
    return mat.flatten('F')

def compute_img_psnr(img_s, img_n):
    a = img_s.astype(np.float64)
    b = img_n.astype(np.float64)
    d = sum((a[:,:,c] - b[:,:,c])**2 for c in range(3))
    mse = np.sum(d) / (3 * a.shape[0] * a.shape[1])
    return mse, (float('inf') if mse == 0 else 10 * np.log10(255**2 / mse))

def img_similarity(a, b):
    aa = matrix_to_row_vector(a).astype(np.float64)
    bb = matrix_to_row_vector(b).astype(np.float64)
    na, nb = np.sqrt(np.dot(aa,aa)), np.sqrt(np.dot(bb,bb))
    return 0.0 if na == 0 or nb == 0 else (np.dot(aa,bb) / nb / na) * 100.0

v1 = cv2.VideoCapture(original_video_path)
v2 = cv2.VideoCapture(cover_output_path)
v3 = cv2.VideoCapture(stego_output_path)
v4 = cv2.VideoCapture(reconstructed_output_path)

num_frames = min(int(v.get(cv2.CAP_PROP_FRAME_COUNT)) for v in [v1,v2,v3,v4])
print(f"Computing metrics over {num_frames} frames...")

results = []
for kk in range(num_frames):
    r1,f1 = v1.read(); r2,f2 = v2.read()
    r3,f3 = v3.read(); r4,f4 = v4.read()
    if not all([r1,r2,r3,r4]):
        print(f"Warning: stopped at frame {kk+1}")
        break

    tgt = (f1.shape[1], f1.shape[0])
    if f2.shape[:2] != f1.shape[:2]: f2 = cv2.resize(f2, tgt)
    if f3.shape[:2] != f1.shape[:2]: f3 = cv2.resize(f3, tgt)
    if f4.shape[:2] != f1.shape[:2]: f4 = cv2.resize(f4, tgt)

    mse_s, psnr_s = compute_img_psnr(f2, f3)
    mse_e, psnr_e = compute_img_psnr(f1, f4)
    sim_s  = img_similarity(f2, f3)
    sim_e  = img_similarity(f1, f4)

    results.append({
        'Frame': kk+1,
        'MSE (Stego)': mse_s,   'PSNR (Stego)': psnr_s,   'Similarity (Stego)': sim_s,
        'MSE (Extracted)': mse_e, 'PSNR (Extracted)': psnr_e, 'Similarity (Extracted)': sim_e,
    })
    if (kk+1) % 30 == 0:
        print(f"  Frame {kk+1}: Stego PSNR={psnr_s:.2f}dB | Extracted PSNR={psnr_e:.2f}dB")

for v in [v1,v2,v3,v4]:
    v.release()

df = pd.DataFrame(results)
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f"\n✓ Results → {output_excel}")

print("\n" + "="*60)
print("AVERAGE RESULTS")
print("="*60)
for label, mk, pk, sk in [
    ("Stego quality  (Cover vs Stego)",        'MSE (Stego)',     'PSNR (Stego)',     'Similarity (Stego)'),
    ("Extraction     (Original vs Reconstructed)", 'MSE (Extracted)', 'PSNR (Extracted)', 'Similarity (Extracted)'),
]:
    print(f"\n{label}:")
    print(f"  Avg MSE:        {np.mean([r[mk] for r in results]):.4f}")
    print(f"  Avg PSNR:       {np.mean([r[pk] for r in results]):.4f} dB")
    print(f"  Avg Similarity: {np.mean([r[sk] for r in results]):.4f}%")
print("="*60)

from google.colab import files
files.download(output_excel)
print("\n✓ Done!")
